# 11주차 — JOIN — 테이블 연결
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. INNER JOIN

In [ ]:
# 내원 + 환자 정보
q("""
SELECT v.visit_id, v.visit_date, v.department, p.sex, p.birth_date
FROM visit v
JOIN patient p ON v.patient_id = p.patient_id
LIMIT 8
""")

In [ ]:
# 3테이블: 내원 + 환자 + 담당의
q("""
SELECT v.visit_id, v.visit_date, p.sex AS 환자성별, s.role, s.department AS 소속과
FROM visit v
JOIN patient p ON v.patient_id = p.patient_id
JOIN staff s ON v.attending_id = s.staff_id
WHERE v.visit_type = '입원' ORDER BY v.visit_date DESC LIMIT 8
""")

## 2. 대표 질문 — 당뇨 환자들의 HbA1c

In [ ]:
# 당뇨(E11) 환자의 모든 HbA1c 결과
q("""
SELECT DISTINCT v.patient_id, l.result_value AS hba1c, l.collected_at
FROM diagnosis d
JOIN visit v  ON d.visit_id = v.visit_id
JOIN visit v2 ON v2.patient_id = v.patient_id
JOIN lab l    ON l.visit_id = v2.visit_id
WHERE d.kcd_code LIKE 'E11%' AND l.test_name = 'HbA1c'
ORDER BY v.patient_id, l.collected_at
""")

## 3. LEFT JOIN — 없는 사람이 보인다

In [ ]:
# 당뇨 환자 전원 + HbA1c 여부 (NULL = 미검사)
q("""
SELECT dm.patient_id, l.result_value AS hba1c
FROM (SELECT DISTINCT v.patient_id
      FROM diagnosis d JOIN visit v ON d.visit_id = v.visit_id
      WHERE d.kcd_code LIKE 'E11%') dm
LEFT JOIN visit v2 ON v2.patient_id = dm.patient_id
LEFT JOIN lab l ON l.visit_id = v2.visit_id AND l.test_name = 'HbA1c'
GROUP BY dm.patient_id
""")

In [ ]:
# HbA1c를 한 번도 안 받은 당뇨 환자
q("""
SELECT DISTINCT v.patient_id
FROM diagnosis d JOIN visit v ON d.visit_id = v.visit_id
WHERE d.kcd_code LIKE 'E11%'
  AND v.patient_id NOT IN (
    SELECT v2.patient_id FROM lab l
    JOIN visit v2 ON l.visit_id = v2.visit_id
    WHERE l.test_name = 'HbA1c')
""")

## 연습문제

In [ ]:
# [연습 Q1] 각 진단(dx_name)에 해당 환자의 성별을 붙여 출력하라
# 힌트: diagnosis→visit→patient
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 뇌 MRI를 찍은 내원의 환자ID·촬영일·판독문을 출력하라
# 힌트: imaging + visit
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 영상 검사를 한 번도 안 받은 환자는 몇 명인가?
# 힌트: LEFT JOIN … IS NULL 또는 NOT IN
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 폐렴(J18) 내원에서 나간 약물 목록을 만들라
# 힌트: diagnosis + prescription, 내원 기준
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 진단+성별**
```sql
SELECT d.dx_name, p.sex FROM diagnosis d
JOIN visit v ON d.visit_id = v.visit_id
JOIN patient p ON v.patient_id = p.patient_id;
```

**A2. 뇌 MRI**
```sql
SELECT v.patient_id, i.taken_date, i.report FROM imaging i
JOIN visit v ON i.visit_id = v.visit_id
WHERE i.modality = 'MRI' AND i.body_part = '뇌';
```

**A3. 영상 없는 환자 수**
```sql
SELECT COUNT(*) FROM patient p
WHERE p.patient_id NOT IN (
  SELECT v.patient_id FROM imaging i JOIN visit v ON i.visit_id = v.visit_id);
```

**A4. 폐렴 약물**
```sql
SELECT DISTINCT rx.drug_name FROM diagnosis d
JOIN prescription rx ON rx.visit_id = d.visit_id
WHERE d.kcd_code LIKE 'J18%';
```